In [ ]:
%config InlineBackend.figure_format = 'retina'

import json
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import os
import pandas as pd
import re
import seaborn as sns

from dotenv import load_dotenv
from matplotlib.patches import Patch
from openinference.semconv.trace import (
    OpenInferenceSpanKindValues,
    SpanAttributes
)
from tqdm.notebook import tqdm

load_dotenv()

SPAN_KIND = SpanAttributes.OPENINFERENCE_SPAN_KIND
OUTPUT_VALUE = SpanAttributes.OUTPUT_VALUE
AGENT = OpenInferenceSpanKindValues.AGENT.value

In [ ]:
original_input_path = "../../data/frames/llm_frames_results_all_ctx_40960_kvq_f16.csv"
original_input = pd.read_csv(original_input_path)
questions = original_input["Prompt"]
answers = original_input["Answer"]

base_path = "../../logs/frames_ollama_qwen3_30b"
output_file = base_path.split("/")[-1]
max_folder = max(int(f) for f in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, f)))
data = []

for i in range(max_folder + 1):
    path = f"{base_path}/{i}"
    for run_idx in range(1):
        run_path = f"{path}/run_{run_idx}"
        summary_path = f"{run_path}/analysis/summary.json"
        step_summary_path = f"{run_path}/analysis/step_summary.json"

        with open(summary_path, "r") as f:
            summary = json.load(f)
        with open(step_summary_path, "r") as f:
            step_summary = json.load(f)

        step_data = {}
        step_idx = 0
        for step in step_summary:
            if step["kind"] != "LLM":
                continue
            for k, v in step.items():
                step_data[f"step_{step_idx}_{k}"] = v
            step_idx += 1

        agent_output = summary["agent_output"]
        if agent_output is not None and "<think>" in agent_output and "</think>" in agent_output:
            summary["agent_output"] = agent_output[agent_output.find("</think>")+len("</think>"):].strip()

        data.append({
            "question": questions[i],
            "answer": answers[i],
            **summary,
            **step_data,
        })

data = pd.DataFrame.from_dict(data)
data.to_csv(f"../../data/frames/profile_results_{output_file}.csv", index=False)

In [ ]:
acc = []
energy_usage = []

inputs = {
    "Qwen 3 1.7B": "../../data/frames/frames_profile_results_qwen3_1.7b_judged.csv",
    "Qwen 3 4B": "../../data/frames/frames_profile_results_qwen3_4b_judged.csv",
    "Qwen 3 8B", "../../data/frames/frames_profile_results_qwen3_8b_judged.csv",
    "Comcade 1.7B to 14B": "../../data/frames/frames_profile_results_qwen3_fixed_cascade_compress_1.7b_to_14b_step_1_judged.csv",
    "Qwen 3 14B": "../../data/frames/frames_profile_results_qwen3_14b_judged.csv",
    "Qwen 3 30B A3B Instruct 2507": "../../data/frames/frames_profile_results_qwen3_30b_judged.csv",
    "Qwen 3 32B": "../../data/frames/frames_profile_results_qwen3_32b_judged.csv",
}

for _, path in inputs.items():
    data = pd.read_csv(path)
    acc.append(round((data["agent_output_eval"] == "CORRECT").mean(), 2))
    energy_usage.append(data["energy_total_mWh"])

In [ ]:
labels = list(inputs.keys())
colors = sns.color_palette("Set2", n_colors=len(labels))

plt.figure(figsize=(8, 5))
handles = []

for i in range(len(energy_usage)):
    bp = plt.boxplot(
        [energy_usage[i]],
        vert=False,
        positions=[acc[i]],
        widths=0.01,
        showfliers=False,
        patch_artist=True
    )

    # Only color the box body
    box = bp['boxes'][0]
    box.set_facecolor(colors[i])
    box.set_linewidth(1)  # thinner border
    # Customize median line
    median = bp['medians'][0]
    median.set_color('black')

    # Add to legend
    handles.append(Patch(facecolor=colors[i], label=labels[i]))

# Final formatting
ax = plt.gca()
ax.set_yticks([])  # wipe all old tick positions
ax.set_yticklabels([])  # wipe all old labels
yticks = np.arange(np.floor(min(acc) * 20) / 20, np.ceil(max(acc) * 20) / 20 + 0.001, 0.05)
ax.set_ylim(np.min(yticks) - 0.03, np.max(yticks) + 0.03)
ax.set_yticks(yticks)
ax.set_yticklabels([f"{y:.2f}" for y in yticks])

plt.grid(True, axis='x')
plt.xlabel("Energy Usage (mWh)")
plt.ylabel("Accuracy")
plt.title("Accuracy vs Energy Usage on FRAMES")
plt.legend(handles=handles, title="Models")
plt.tight_layout()
plt.savefig("../../figures/ollama_frames.png", dpi=300)
plt.show()